In [7]:
import os
import json
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm import tqdm


def get_config(dataset: str) -> dict:
    if dataset == "mall":
        return {
            "data_dir": "./mall",
            "date_list": ["2024-03-28", "2024-03-29", "2024-03-31", "2024-05-29", "2024-05-31", "2024-06-03"],
            "metrics_list": {
                "host": ["cpu", "disk", "memory", "network"],
                "jvm": ["gc_count_instant", "gc_time_instant", "heap", "thread"],
                "pod": ["cpu", "file_system", "memory", "network", "process"],
            },
            "outer_name": [
                "Timestamp", "timestamp", "IpAddress", "ServiceName", "OperationName", "RpcName",
                "start_time", "end_time", "TraceID", "SpanId", "ParentSpanId", "ServiceIp", "center_timestamp"
            ],
            "output_dir": "../mall",
        }
    elif dataset == "train-ticket":
        return {
            "data_dir": "./train-ticket",
            "date_list": ["2024-06-25", "2024-06-26", "2024-06-27", "2024-06-27-1", "2024-06-28", "2024-07-01"],
            "metrics_list": {
                "host": ["cpu", "disk", "memory", "network"],
                "jvm": ["gc_count_instant", "gc_time_instant", "heap", "thread"],
                "pod": ["cpu", "file_system", "memory", "network", "process"],
            },
            "outer_name": [
                "Timestamp", "timestamp", "IpAddress", "ServiceName", "OperationName", "RpcName",
                "start_time", "end_time", "TraceID", "SpanId", "ParentSpanId", "ServiceIp", "center_timestamp"
            ],
            "output_dir": "../train-ticket",
        }
    else:
        raise ValueError("dataset must be 'mall' or 'train-ticket'")


class MetricsMerger:
    """
    作用：
    1. 扫描所有 metrics csv，收集全量 timestamp / IpAddress / 指标列名
    2. 构建一个以 (IpAddress, timestamp) 为索引的宽表
    3. 把 host/jvm/pod 各类指标填进去
    4. 做缺失值填充
    5. 导出 ip_metrics.csv
    """

    def __init__(self, config: dict):
        self.config = config
        self.data_dir = config["data_dir"]
        self.date_list = config["date_list"]
        self.metrics_list = config["metrics_list"]
        self.outer_name = config["outer_name"]
        self.output_dir = config["output_dir"]

        self.ip_metrics_name: List[str] = []
        self.timestamps: List[int] = []
        self.ip_addresses: List[str] = []

        self.ip_metrics_df: pd.DataFrame | None = None

    @staticmethod
    def add_duplicate(raw_list: list, add_content: list) -> list:
        """原项目里的 add_duplicate：做去重并合并。"""
        raw_list += list(set(add_content))
        raw_list = list(set(raw_list))
        return raw_list

    def build_metric_file_path(self, date: str, metrics_dir: str, metrics_name: str) -> str:
        return os.path.join(self.data_dir, date, "metrics", metrics_dir, f"{metrics_name}.csv")

    def scan_global_index_and_columns(self) -> None:
        """
        第一遍扫描：
        - 收集所有时间戳 timestamps
        - 收集所有 IP 地址 ip_addresses
        - 收集所有指标列名 ip_metrics_name
        """
        for date in tqdm(self.date_list, postfix="scan metric schema"):
            for metrics_dir, metrics_name_list in self.metrics_list.items():
                for metrics_name in metrics_name_list:
                    metrics_file_url = self.build_metric_file_path(date, metrics_dir, metrics_name)
                    metrics_df = pd.read_csv(metrics_file_url)
                    '''
                        timestamp  SystemCpuSystem  SystemCpuIOWait  SystemCpuUser   IpAddress
                        0  1711555200             0.89             0.00           1.57  10.0.0.166
                        1  1711555205             0.89             0.00           1.57  10.0.0.166
                    '''

                    # 把当前 metrics 文件里的 timestamp 加入“全局时间集合”
                    self.timestamps = self.add_duplicate(
                        self.timestamps,
                        [int(item) for item in metrics_df["timestamp"].to_list()]
                    )

                    # 把当前文件里的 IP 加入“全局 IP 集合”
                    self.ip_addresses = self.add_duplicate(
                        self.ip_addresses,
                        metrics_df["IpAddress"].to_list()
                    )

                    for column_name in metrics_df.columns.to_list():
                        # 过滤掉非指标的公共字段
                        if column_name not in self.outer_name:
                            # host/pod/jvm_指标类名_原始指标名
                            merged_name = f"{metrics_dir}_{metrics_name}_{column_name}"

                            if merged_name not in self.ip_metrics_name:
                                self.ip_metrics_name.append(merged_name)

    def build_empty_metric_table(self) -> pd.DataFrame:
        """
        构建一张空表：
        行 = 所有 IP × 所有 timestamp
        列 = timestamp, IpAddress, 所有统一命名后的 metric 列
        """
        self.timestamps = sorted(self.timestamps)

        # 给每个ip构造了一段时间戳
        ip_metrics_dict = {
            "timestamp": self.timestamps * len(self.ip_addresses),
            "IpAddress": [],
        }

        # 每个ip重复timestamps次，相当于和timestamp对齐了
        for ip_address in self.ip_addresses:
            ip_metrics_dict["IpAddress"] += [ip_address] * len(self.timestamps)

        total_rows = len(ip_metrics_dict["timestamp"])
        for ip_name in self.ip_metrics_name:
            # host/pod/jvm_指标类名_原始指标名
            ip_metrics_dict[ip_name] = [np.nan] * total_rows

        ''' 
            构建一个类似的表
            timestamp	IpAddress	host_cpu_usage	pod_memory_usage
            1	A	NaN	NaN
            2	A	NaN	NaN
            3	A	NaN	NaN
            1	B	NaN	NaN
            2	B	NaN	NaN
            3	B	NaN	NaN
        '''
        self.ip_metrics_df = pd.DataFrame(ip_metrics_dict)
        return self.ip_metrics_df

    def fill_metric_values(self) -> pd.DataFrame:
        """
        第二遍扫描：
        把每个 csv 的数值填入大表。
        通过 (IpAddress, timestamp) -> 行号 的方式快速定位。
        """
        if self.ip_metrics_df is None:
            raise ValueError("Please run build_empty_metric_table() first.")

        ip_2_idx = {ip: idx * len(self.timestamps) for idx, ip in enumerate(self.ip_addresses)}
        timestamp_2_idx = {ts: idx for idx, ts in enumerate(self.timestamps)}

        for date in tqdm(self.date_list, postfix="merge metric values"):
            for metrics_dir, metrics_name_list in self.metrics_list.items():
                for metrics_name in metrics_name_list:
                    metrics_file_url = self.build_metric_file_path(date, metrics_dir, metrics_name)
                    metrics_df = pd.read_csv(metrics_file_url)

                    raw_metric_columns = [
                        col for col in metrics_df.columns.to_list()
                        if col not in self.outer_name
                    ]
                    merged_metric_columns = [
                        f"{metrics_dir}_{metrics_name}_{col}"
                        for col in raw_metric_columns
                    ]

                    line_ids = [
                        ip_2_idx[row_ip] + timestamp_2_idx[row_ts]
                        for row_ip, row_ts in zip(
                            metrics_df["IpAddress"].to_list(),
                            metrics_df["timestamp"].to_list()
                        )
                    ]

                    self.ip_metrics_df.loc[line_ids, merged_metric_columns] = metrics_df.loc[
                        :, raw_metric_columns
                    ].values

        return self.ip_metrics_df

    def nan_2_num(self, metrics_df: pd.DataFrame) -> pd.DataFrame:
        """
        原项目里的缺失值处理逻辑：

        1. 先做前向填充 ffill
        2. 包含 count / exception 的列，用 0 填
        3. 其他指标列，用列均值填
        """
        # 原代码里是从 outer_name 里挑第二个，通常是 'timestamp'
        df_index = [col for col in metrics_df.columns.to_list() if col in self.outer_name][1]

        # 给每个 IP 的第一条记录预留位置，避免直接被错误 ffill
        first_line = [
            i + len(set(metrics_df["timestamp"].to_list()))
            for i in range(len(set(metrics_df[df_index].to_list())))
        ]
        valid_first_line = [i for i in first_line if i < len(metrics_df)]

        if len(valid_first_line) > 0:
            metrics_df.loc[valid_first_line] = metrics_df.loc[valid_first_line].fillna(value=pd.NA)

        metrics_df.fillna(method="ffill", inplace=True)
        metrics_df.replace({pd.NA: np.nan}, inplace=True)

        cols_with_keywords = [
            col for col in metrics_df.columns
            if ("count" in col or "exception" in col)
        ]
        if cols_with_keywords:
            metrics_df[cols_with_keywords] = metrics_df[cols_with_keywords].fillna(0)

        metric_columns = [
            col for col in metrics_df.columns.to_list()
            if col not in self.outer_name
        ]
        metrics_df[metric_columns] = metrics_df[metric_columns].apply(
            lambda col: col.fillna(col.mean()),
            axis=0
        )
        return metrics_df

    def save(self, filename: str = "ip_metrics.csv") -> str:
        if self.ip_metrics_df is None:
            raise ValueError("No merged dataframe to save.")
        output_path = os.path.join(self.output_dir, filename)
        '''
            timestamp	IpAddress	host_cpu_usage	pod_memory_usage
            1000	10.0.0.1	0.5	0.7
            1001	10.0.0.1	0.6	0.8
            1000	10.0.0.2	0.4	0.6
        '''
        self.ip_metrics_df.to_csv(output_path, index=False)
        print(f"save merged metrics to: {output_path}")
        return output_path

    def run(self) -> pd.DataFrame:
        self.scan_global_index_and_columns()
        self.build_empty_metric_table()
        # 填数值
        self.fill_metric_values()
        self.ip_metrics_df = self.nan_2_num(self.ip_metrics_df)
        self.save()
        return self.ip_metrics_df

In [9]:

# 直接改这里即可
dataset = "train-ticket"   # 或 "train-ticket"

config = get_config(dataset)

merger = MetricsMerger(config)
merged_df = merger.run()

print("\nmerged_df shape:", merged_df.shape)
print(merged_df.head())

100%|██████████| 6/6 [00:01<00:00,  4.12it/s, merge metric values]
/tmp/ipykernel_3438348/2543704214.py:216: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  metrics_df.fillna(method="ffill", inplace=True)


save merged metrics to: ../train-ticket/ip_metrics.csv

merged_df shape: (1060884, 39)
    timestamp   IpAddress  host_cpu_SystemCpuSystem  host_cpu_SystemCpuIOWait  \
0  1719300600  10.2.0.245                  3.613433                  1.676091   
1  1719300605  10.2.0.245                  3.613433                  1.676091   
2  1719300610  10.2.0.245                  3.613433                  1.676091   
3  1719300615  10.2.0.245                  3.613433                  1.676091   
4  1719300620  10.2.0.245                  3.613433                  1.676091   

   host_cpu_SystemCpuUser  host_disk_SystemDiskFree  host_disk_SystemDiskUsed  \
0                9.208508              1.005249e+11              2.589896e+10   
1                9.208508              1.005249e+11              2.589896e+10   
2                9.208508              1.005249e+11              2.589896e+10   
3                9.208508              1.005249e+11              2.589896e+10   
4                9.20